# Problem 2 — Churn Prediction on Imbalanced Data

**Goal:** using `churn_labels.csv` as the target, build a model that identifies customers at
risk of leaving and reports performance in terms a retention team can act on.

**Key constraints driving every decision below:**
- Churn is a rare event (~4% of customers). Accuracy is not used as the primary metric.
- Every feature is checked for leakage before it is used: *"would this information actually
  have been available at the moment we needed to make the prediction?"*
- The dataset is messy on purpose (mixed date formats, inconsistent keys, bad values). We do
  not silently drop rows — every non-trivial data problem is documented as
  `PROBLEM / EVIDENCE / TREATMENT / WHY`.
- No filenames are hard-coded blindly: the notebook discovers the CSVs in the folder and
  classifies them by their columns, so it keeps working even if a filename differs slightly
  from what we expect.

# 2. Imports and Configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

import re
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss,
)
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# --- EDIT THIS PATH IF NEEDED -------------------------------------------------
DATA_DIR = Path(r"C:\Users\LENOVO\Downloads\Precisely_Hackathon_Datasets")
# -------------------------------------------------------------------------------

# There is no explicit "prediction cutoff" column anywhere in the data (churn_labels.csv has
# no date). We therefore need a single reference "as of today" date to compute tenure /
# recency features consistently. We use the current real-world date. This is stated explicitly
# because it is a modeling assumption, not a fact read from the data.
SNAPSHOT_DATE = pd.Timestamp("2026-09-18")

print("Data directory:", DATA_DIR)
print("Snapshot date used for tenure / recency features:", SNAPSHOT_DATE.date())

Data directory: C:\Users\LENOVO\Downloads\Precisely_Hackathon_Datasets
Snapshot date used for tenure / recency features: 2026-09-18


### Helper functions

The raw files mix several date formats in the *same column*
(`27 Mar 2025`, `16-11-2019`, `2020-05-04`, `2026-06-01T00:00:00Z` ...), inconsistent email
casing, and non-numeric values stored in numeric columns (e.g. `"-"` for resolution hours,
`"ABC123"` for a pincode). These helpers centralize robust, reusable parsing so every section
below treats messiness the same way instead of ad-hoc fixes scattered through the notebook.

In [2]:
def parse_dates_robust(series):
    '''Parse a column containing mixed date formats: dd-mm-yyyy, dd/mm/yyyy, "DD Mon YYYY",
    ISO yyyy-mm-dd, and ISO datetimes with a trailing "Z" (UTC). ISO-formatted strings
    (yyyy-mm-dd...) are parsed separately and NOT run through dayfirst=True, because that
    combination silently corrupts unambiguous ISO dates (e.g. turns 2025-09-02 into 2025-02-09).
    Non-ISO strings are parsed with dayfirst=True since this data uses DD-MM-YYYY /
    DD/MM/YYYY conventions. Anything that still cannot be parsed becomes NaT (missing).'''
    s = series.astype(str).str.strip()
    s = s.replace({"nan": np.nan, "NaT": np.nan, "None": np.nan, "": np.nan})
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")

    iso_mask = s.str.match(r"^\d{4}-\d{2}-\d{2}").fillna(False)
    if iso_mask.any():
        iso_parsed = pd.to_datetime(s[iso_mask], errors="coerce", utc=True, format="mixed")
        result.loc[iso_mask] = iso_parsed.dt.tz_localize(None)

    other_mask = (~iso_mask) & s.notna()
    if other_mask.any():
        other_parsed = pd.to_datetime(s[other_mask], errors="coerce", dayfirst=True, format="mixed")
        result.loc[other_mask] = other_parsed

    return result

def normalize_email(series):
    out = series.astype(str).str.strip().str.lower()
    return out.replace({"nan": np.nan, "none": np.nan, "": np.nan})

def normalize_text(series):
    out = series.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    return out.replace({"nan": np.nan, "": np.nan})

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def looks_like_email(value):
    if pd.isna(value):
        return False
    return bool(re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", str(value).strip()))

CRM_ID_PATTERN = re.compile(r"^CRM-\d+$")

# 3. Dataset Discovery

We do not assume filenames or a fixed CRM/billing/support/churn structure. Every CSV in the
folder is loaded, profiled, and then classified by its actual columns.

In [3]:
csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR}. Check DATA_DIR above.")

raw_frames = {}
discovery_rows = []

for f in csv_files:
    df_tmp = pd.read_csv(f, low_memory=False)
    raw_frames[f.name] = df_tmp
    discovery_rows.append({
        "file": f.name,
        "rows": len(df_tmp),
        "cols": df_tmp.shape[1],
        "missing_cells": int(df_tmp.isna().sum().sum()),
        "pct_missing": round(df_tmp.isna().sum().sum() / df_tmp.size * 100, 2),
        "duplicate_rows": int(df_tmp.duplicated().sum()),
    })

discovery_df = pd.DataFrame(discovery_rows)
display(discovery_df)

for f in csv_files:
    print(f"\n{f.name} columns -> {list(raw_frames[f.name].columns)}")

,file,rows,cols,missing_cells,pct_missing,duplicate_rows
0,churn_labels.csv,4750,2,0,0.00,0
1,customers_billing.csv,4050,14,995,1.75,0
2,customers_crm.csv,4750,12,997,1.75,0
3,customers_support.csv,3733,11,876,2.13,0



churn_labels.csv columns -> ['cust_id', 'churned']

customers_billing.csv columns -> ['billing_id', 'customer_name', 'contact_email', 'contact_phone', 'billing_city', 'billing_state', 'postal_code', 'plan_type', 'monthly_amount', 'currency', 'last_payment_date', 'payment_status', 'tenure_months', 'region_office']

customers_crm.csv columns -> ['cust_id', 'full_name', 'email', 'phone', 'address_line', 'city', 'state', 'pincode', 'signup_date', 'segment', 'account_status', 'region_office']

customers_support.csv columns -> ['ticket_id', 'cust_ref', 'reporter_name', 'reporter_email', 'ticket_opened', 'ticket_closed', 'severity', 'category', 'resolution_hours', 'satisfaction_score', 'region_office']


In [4]:
def profile_file(name, df):
    print("=" * 100)
    print(f"FILE: {name}   shape={df.shape}")
    info = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(dropna=True),
    })
    display(info)
    display(df.sample(min(3, len(df)), random_state=RANDOM_SEED))

for name, df in raw_frames.items():
    profile_file(name, df)

FILE: churn_labels.csv   shape=(4750, 2)


,dtype,missing,missing_%,n_unique
cust_id,object,0,0.0,4750
churned,int64,0,0.0,2


,cust_id,churned
3817,CRM-00431,0
1075,CRM-02620,0
296,CRM-03174,0


FILE: customers_billing.csv   shape=(4050, 14)


,dtype,missing,missing_%,n_unique
billing_id,object,0,0.0,4050
customer_name,object,12,0.3,1671
contact_email,object,180,4.4,3539
contact_phone,object,240,5.9,3642
billing_city,object,121,3.0,8
billing_state,object,152,3.8,20
postal_code,object,182,4.5,696
plan_type,object,108,2.7,6
monthly_amount,float64,0,0.0,3832
currency,object,0,0.0,2


...

,dtype,missing,missing_%,n_unique
cust_id,object,0,0.0,4750
full_name,object,17,0.4,1853
email,object,161,3.4,4270
phone,object,210,4.4,4437
address_line,object,0,0.0,2161
city,object,107,2.3,37
state,object,127,2.7,20
pincode,object,158,3.3,706
signup_date,object,0,0.0,4031
segment,object,217,4.6,6


,cust_id,full_name,email,phone,address_line,city,state,pincode,signup_date,segment,account_status,region_office
3817,CRM-03818,Aarti Rao,arao43@outlook.com,07886565916,"45, Baner Road",NaN,Tamil Nadu,,2023-08-27T00:00:00Z,Startup,Active,Chennai
1075,CRM-01076,Lakshmi Naidu,lakshmi866@rediffmail.com,76016 79749,"238, Park Street",Pune,Maharashtra,411021,23-07-2024,Enterprise,Active,Pune
296,CRM-00297,Kunal Bos,kbose6@rediffmail.com,06469902340,"212, Anna Salai",Gurugram,HR,12205,2021-07-04,SMB,Active,Gurugram


FILE: customers_support.csv   shape=(3733, 11)


,dtype,missing,missing_%,n_unique
ticket_id,object,0,0.0,3733
cust_ref,object,75,2.0,2453
reporter_name,object,0,0.0,1292
reporter_email,object,243,6.5,1737
ticket_opened,object,0,0.0,2938
ticket_closed,object,193,5.2,2744
severity,object,0,0.0,4
category,object,0,0.0,8
resolution_hours,object,127,3.4,826
satisfaction_score,object,238,6.4,12


,ticket_id,cust_ref,reporter_name,reporter_email,ticket_opened,ticket_closed,severity,category,resolution_hours,satisfaction_score,region_office
1800,TKT-000690,priya.agarwal@yahoo.in,Priya Agarwal,priya.agarwal@yahoo.in,14/04/2026,2026-04-16T00:00:00Z,High,Billing Query,50.0,1,Bengaluru
3188,TKT-001921,divya600@corpmail.co.in,Divya Chauhan,divya600@corpmail.co.in,2025-11-15,NaN,Medium,Feature Request,18.6,3,Chennai
1805,TKT-002715,asharma4@yahoo.in,Ananya Sharma,asharma4@yahoo.in,13-06-2023,14 Jun 2023,Medium,Data Sync Failure,18.9,1,Gurugram


In [5]:
# Classify each file by the columns it actually has, rather than by filename.
def classify(df):
    cols = set(df.columns)
    if "churned" in cols:
        return "churn"
    if {"cust_id", "full_name"}.issubset(cols):
        return "crm"
    if "billing_id" in cols:
        return "billing"
    if "ticket_id" in cols:
        return "support"
    return "unknown"

roles = {name: classify(df) for name, df in raw_frames.items()}
print(roles)

role_to_files = {}
for name, role in roles.items():
    role_to_files.setdefault(role, []).append(name)

for role in ["churn", "crm", "billing", "support"]:
    files = role_to_files.get(role, [])
    if len(files) != 1:
        raise ValueError(
            f"Expected exactly 1 file for role '{role}', found {files}. "
            f"Inspect `roles` above -- a filename/column may not match the classify() rule."
        )

df_churn = raw_frames[role_to_files["churn"][0]].copy()
df_crm = raw_frames[role_to_files["crm"][0]].copy()
df_billing = raw_frames[role_to_files["billing"][0]].copy()
df_support = raw_frames[role_to_files["support"][0]].copy()

print("churn file  ->", role_to_files["churn"][0])
print("crm file    ->", role_to_files["crm"][0])
print("billing file->", role_to_files["billing"][0])
print("support file->", role_to_files["support"][0])

{'churn_labels.csv': 'churn', 'customers_billing.csv': 'billing', 'customers_crm.csv': 'crm', 'customers_support.csv': 'support'}
churn file  -> churn_labels.csv
crm file    -> customers_crm.csv
billing file-> customers_billing.csv
support file-> customers_support.csv


# 4. Data Quality Audit

For each file we check the issue types the brief calls out explicitly (duplicates, missing
IDs, impossible values, casing, negative amounts, bad dates) and document the treatment. We
never silently delete rows — invalid values become `NaN`/`NaT` (missing) so they flow through
the pipeline's imputers, and every fix is explained below.

### CRM

**PROBLEM:** `pincode` sometimes contains non-numeric junk (e.g. `"ABC123"`); `signup_date`
mixes several date formats; `state`/`city` have inconsistent casing and blank strings.
**EVIDENCE:** printed below.
**TREATMENT:** `pincode` -> numeric coercion (bad values become missing); dates -> robust
mixed-format parser; text fields -> stripped + title-cased, blank strings treated as missing.
**WHY:** these fields are not modeling-critical on their own, but a bad `pincode`/`state`
would silently corrupt any geography feature, and a corrupted `signup_date` corrupts tenure —
one of our strongest expected features.

In [6]:
df_crm_c = df_crm.copy()

for col in ["full_name", "email", "city", "state", "segment", "account_status", "region_office"]:
    if col in df_crm_c:
        df_crm_c[col] = normalize_text(df_crm_c[col])

df_crm_c["email"] = normalize_email(df_crm_c["email"])
for col in ["city", "state", "segment", "account_status", "region_office"]:
    if col in df_crm_c:
        df_crm_c[col] = df_crm_c[col].str.title()

pincode_numeric = safe_numeric(df_crm_c["pincode"])
print("CRM: invalid (non-numeric) pincodes ->", pincode_numeric.isna().sum() - df_crm_c["pincode"].isna().sum())
df_crm_c["pincode"] = pincode_numeric

df_crm_c["signup_date"] = parse_dates_robust(df_crm_c["signup_date"])
print("CRM: signup_date values that failed to parse ->", df_crm_c["signup_date"].isna().sum())
print("CRM: signup_date in the future vs snapshot date ->", (df_crm_c["signup_date"] > SNAPSHOT_DATE).sum())
df_crm_c.loc[df_crm_c["signup_date"] > SNAPSHOT_DATE, "signup_date"] = pd.NaT  # impossible: can't sign up in the future

dup_cust_ids = df_crm_c["cust_id"].duplicated().sum()
print("CRM: duplicate cust_id rows ->", dup_cust_ids)
df_crm_c = df_crm_c.drop_duplicates(subset="cust_id", keep="first")

const_cols = [c for c in df_crm_c.columns if df_crm_c[c].nunique(dropna=True) <= 1]
print("CRM: constant/near-constant columns ->", const_cols)

CRM: invalid (non-numeric) pincodes -> 106
CRM: signup_date values that failed to parse -> 0
CRM: signup_date in the future vs snapshot date -> 36
CRM: duplicate cust_id rows -> 0
CRM: constant/near-constant columns -> []


### Billing

**PROBLEM:** `monthly_amount` contains negative values (e.g. `-8866.68`), which is not a
sensible value for a recurring subscription charge; `last_payment_date` mixes local formats
and ISO-8601-with-`Z`; there is **no direct `cust_id` column** — billing must be linked to CRM
through `contact_email`, and some emails are missing.
**EVIDENCE:** printed below.
**TREATMENT:** negative amounts are flagged and excluded from the *clean* amount used for
modeling (kept in an `amount_negative_flag` column for audit); dates parsed with the mixed
parser; billing rows with no usable email are kept in the raw table but simply won't join to a
customer (reported as an unmatched count in Section 5, not deleted).
**WHY:** a raw average that includes negative "revenue" would make `monthly_amount_mean`
meaningless and could look like a spuriously strong (and wrong) predictor.

In [7]:
df_billing_c = df_billing.copy()

for col in ["customer_name", "billing_city", "billing_state", "plan_type", "payment_status"]:
    if col in df_billing_c:
        df_billing_c[col] = normalize_text(df_billing_c[col])
df_billing_c["contact_email"] = normalize_email(df_billing_c["contact_email"])
for col in ["billing_state", "plan_type", "payment_status"]:
    if col in df_billing_c:
        df_billing_c[col] = df_billing_c[col].str.title()

df_billing_c["monthly_amount"] = safe_numeric(df_billing_c["monthly_amount"])
df_billing_c["amount_negative_flag"] = (df_billing_c["monthly_amount"] < 0).astype(int)
print("Billing: negative monthly_amount rows ->", df_billing_c["amount_negative_flag"].sum())
df_billing_c["monthly_amount_clean"] = df_billing_c["monthly_amount"].where(df_billing_c["monthly_amount"] >= 0)

df_billing_c["tenure_months"] = safe_numeric(df_billing_c["tenure_months"])
df_billing_c["last_payment_date"] = parse_dates_robust(df_billing_c["last_payment_date"])
print("Billing: last_payment_date failed to parse ->", df_billing_c["last_payment_date"].isna().sum())
print("Billing: missing contact_email ->", df_billing_c["contact_email"].isna().sum())
print("Billing: duplicate billing_id rows ->", df_billing_c["billing_id"].duplicated().sum())

Billing: negative monthly_amount rows -> 59
Billing: last_payment_date failed to parse -> 0
Billing: missing contact_email -> 216
Billing: duplicate billing_id rows -> 0


### Support

**PROBLEM:** `cust_ref` is inconsistent — sometimes a proper CRM id (`CRM-04211`), sometimes
an email address instead; `resolution_hours` contains the literal string `"-"` for some rows;
`ticket_closed` is sometimes earlier than `ticket_opened` (impossible); dates mix formats and
some ticket dates fall in the future relative to the snapshot date.
**EVIDENCE:** printed below.
**TREATMENT:** `cust_ref` resolution is handled explicitly in Section 5 (relationships), since
it is really a join-quality problem, not just a cleaning problem; `resolution_hours` is
coerced to numeric (`"-"` -> missing); rows where `ticket_closed < ticket_opened` have
`ticket_closed` set to missing (we keep `ticket_opened`, which is still informative, but do not
trust a resolution time that is impossible).
**WHY:** an uncorrected negative/impossible resolution time would silently poison
`avg_resolution_hours` as a feature.

In [8]:
df_support_c = df_support.copy()

for col in ["reporter_name", "severity", "category", "region_office"]:
    if col in df_support_c:
        df_support_c[col] = normalize_text(df_support_c[col])
df_support_c["reporter_email"] = normalize_email(df_support_c["reporter_email"])
for col in ["severity", "category"]:
    if col in df_support_c:
        df_support_c[col] = df_support_c[col].str.title()

df_support_c["resolution_hours"] = safe_numeric(df_support_c["resolution_hours"])
df_support_c["satisfaction_score"] = safe_numeric(df_support_c["satisfaction_score"])

df_support_c["ticket_opened"] = parse_dates_robust(df_support_c["ticket_opened"])
df_support_c["ticket_closed"] = parse_dates_robust(df_support_c["ticket_closed"])

impossible_order = df_support_c["ticket_closed"] < df_support_c["ticket_opened"]
print("Support: tickets closed before they opened (impossible) ->", impossible_order.sum())
df_support_c.loc[impossible_order, "ticket_closed"] = pd.NaT

future_open = df_support_c["ticket_opened"] > SNAPSHOT_DATE
print("Support: tickets opened after the snapshot date ->", future_open.sum())
df_support_c.loc[future_open, "ticket_opened"] = pd.NaT

print("Support: resolution_hours non-numeric values coerced to missing ->",
      df_support_c["resolution_hours"].isna().sum())
print("Support: duplicate ticket_id rows ->", df_support_c["ticket_id"].duplicated().sum())

Support: tickets closed before they opened (impossible) -> 227
Support: tickets opened after the snapshot date -> 21
Support: resolution_hours non-numeric values coerced to missing -> 191
Support: duplicate ticket_id rows -> 0


# 5. Dataset Relationships / Customer Keys

`cust_id` (format `CRM-#####`) is CRM's own primary key. Neither billing nor support carries
that key directly:

- **Billing** has no customer id at all — it must be linked through `contact_email`.
- **Support**'s `cust_ref` is unreliable: it is sometimes a real `cust_id`, sometimes an email
  address dropped into the same column by mistake. We resolve it with a fallback chain and
  measure how well each method actually works, rather than assuming it works.

We check uniqueness and overlap before doing any merge, and aggregate billing/support to one
row per customer **before** merging, so a customer with 5 tickets doesn't get duplicated into
5 modeling rows.

In [9]:
crm_email_map = (
    df_crm_c.dropna(subset=["email"])
            .drop_duplicates(subset="email", keep="first")
            .set_index("email")["cust_id"]
)
crm_id_set = set(df_crm_c["cust_id"])

# --- Billing -> CRM, via email -------------------------------------------------
df_billing_c["matched_cust_id"] = df_billing_c["contact_email"].map(crm_email_map)
billing_match_rate = df_billing_c["matched_cust_id"].notna().mean()
print(f"Billing rows matched to a CRM customer via email: {billing_match_rate:.1%} "
      f"({df_billing_c['matched_cust_id'].notna().sum()}/{len(df_billing_c)})")

billing_per_customer = df_billing_c.dropna(subset=["matched_cust_id"]).groupby("matched_cust_id").size()
print("Customers with more than one billing record:", int((billing_per_customer > 1).sum()))

# --- Support -> CRM, via cust_ref then reporter_email fallback ----------------
def resolve_support_customer(row):
    ref = row["cust_ref"]
    if pd.notna(ref):
        ref_str = str(ref).strip()
        if CRM_ID_PATTERN.match(ref_str) and ref_str in crm_id_set:
            return ref_str, "direct_id"
        if looks_like_email(ref_str):
            cid = crm_email_map.get(ref_str.lower())
            if cid is not None:
                return cid, "cust_ref_as_email"
    rep_email = row.get("reporter_email")
    if pd.notna(rep_email):
        cid = crm_email_map.get(str(rep_email).strip().lower())
        if cid is not None:
            return cid, "reporter_email_fallback"
    return np.nan, "unmatched"

resolved = df_support_c.apply(resolve_support_customer, axis=1, result_type="expand")
df_support_c["matched_cust_id"] = resolved[0]
df_support_c["match_method"] = resolved[1]

print("\nSupport ticket -> customer resolution method:")
print(df_support_c["match_method"].value_counts())
print(f"Support tickets matched overall: {df_support_c['matched_cust_id'].notna().mean():.1%}")

tickets_per_customer = df_support_c.dropna(subset=["matched_cust_id"]).groupby("matched_cust_id").size()
print("\nSupport is many-to-one as expected -- customers with >1 ticket:",
      int((tickets_per_customer > 1).sum()), "/ matched customers:", tickets_per_customer.shape[0])

Billing rows matched to a CRM customer via email: 81.2% (3290/4050)
Customers with more than one billing record: 229

Support ticket -> customer resolution method:
match_method
direct_id                  1910
cust_ref_as_email           810
unmatched                   510
reporter_email_fallback     503
Name: count, dtype: int64
Support tickets matched overall: 86.3%

Support is many-to-one as expected -- customers with >1 ticket: 727 / matched customers: 1653


# 6. Target Definition

`churn_labels.csv` is the only source of the target. We do not invent a churn definition —
`churned` is used exactly as given (0/1).

In [10]:
dup_label_ids = df_churn["cust_id"].duplicated().sum()
print("Duplicate cust_id rows in churn_labels ->", dup_label_ids)
df_churn_c = df_churn.drop_duplicates(subset="cust_id", keep="first").copy()
df_churn_c["churned"] = safe_numeric(df_churn_c["churned"]).astype(int)

n = len(df_churn_c)
churners = int(df_churn_c["churned"].sum())
churn_rate = df_churn_c["churned"].mean()
print(f"Customers with a label: {n}")
print(f"Churners: {churners}   Retained: {n - churners}")
print(f"Churn rate: {churn_rate:.2%}")

labels_without_crm = set(df_churn_c["cust_id"]) - crm_id_set
crm_without_label = crm_id_set - set(df_churn_c["cust_id"])
print("\nLabelled customers with no matching CRM profile:", len(labels_without_crm))
print("CRM customers with no churn label (cannot be used for training):", len(crm_without_label))

Duplicate cust_id rows in churn_labels -> 0
Customers with a label: 4750
Churners: 195   Retained: 4555
Churn rate: 4.11%

Labelled customers with no matching CRM profile: 0
CRM customers with no churn label (cannot be used for training): 0


# 7. Customer 360 — Aggregating Billing & Support to One Row per Customer

Billing and support are aggregated to customer level **before** joining to CRM/labels, so the
merge cannot multiply rows. The final table is verified to have exactly one row per `cust_id`.

In [11]:
billing_agg = (
    df_billing_c.dropna(subset=["matched_cust_id"])
    .groupby("matched_cust_id")
    .agg(
        billing_records=("billing_id", "count"),
        monthly_amount_mean=("monthly_amount_clean", "mean"),
        amount_negative_flag_count=("amount_negative_flag", "sum"),
        tenure_months_billing=("tenure_months", "max"),
        last_payment_date=("last_payment_date", "max"),
        payment_failed_count=("payment_status", lambda s: (s == "Failed").sum()),
        plan_type=("plan_type", lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
    )
    .reset_index()
    .rename(columns={"matched_cust_id": "cust_id"})
)
billing_agg["days_since_last_payment"] = (SNAPSHOT_DATE - billing_agg["last_payment_date"]).dt.days

support_agg = (
    df_support_c.dropna(subset=["matched_cust_id"])
    .groupby("matched_cust_id")
    .agg(
        ticket_count=("ticket_id", "count"),
        unresolved_ticket_count=("ticket_closed", lambda s: s.isna().sum()),
        avg_resolution_hours=("resolution_hours", "mean"),
        avg_satisfaction=("satisfaction_score", "mean"),
        high_severity_count=("severity", lambda s: (s == "High").sum()),
        last_ticket_date=("ticket_opened", "max"),
    )
    .reset_index()
    .rename(columns={"matched_cust_id": "cust_id"})
)
support_agg["days_since_last_ticket"] = (SNAPSHOT_DATE - support_agg["last_ticket_date"]).dt.days

customer360 = (
    df_crm_c
    .merge(df_churn_c, on="cust_id", how="inner")   # inner: a label is required to train/evaluate
    .merge(billing_agg, on="cust_id", how="left")
    .merge(support_agg, on="cust_id", how="left")
)

assert customer360["cust_id"].is_unique, "Join produced duplicate customers -- stop and investigate."
print("customer360 shape:", customer360.shape)
print("Customers with no billing record:", customer360["billing_records"].isna().sum())
print("Customers with no support ticket:", customer360["ticket_count"].isna().sum())
display(customer360.head(3))

customer360 shape: (4750, 28)
Customers with no billing record: 1733
Customers with no support ticket: 3097


,cust_id,full_name,email,phone,address_line,city,state,pincode,signup_date,segment,account_status,region_office,churned,billing_records,monthly_amount_mean,amount_negative_flag_count,tenure_months_billing,last_payment_date,payment_failed_count,plan_type,days_since_last_payment,ticket_count,unresolved_ticket_count,avg_resolution_hours,avg_satisfaction,high_severity_count,last_ticket_date,days_since_last_ticket
0,CRM-00001,Tanvi Naidu,tnaidu6@corpmail.co.in,92368 97913,"161, Jubilee Hills",Pune,NaN,411050.0,2025-03-27,Mid-Market,Active,Pune,0,1.0,3234.93,0.0,15.0,2026-06-20,0.0,Standard,90.0,NaN,NaN,NaN,NaN,NaN,NaT,NaN
1,CRM-00002,Nandini Shah,nandini124@rediffmail.com,(9460) 165328,"98, Indiranagar",Hyderabad,Telangana,500049.0,2019-11-16,Smb,Active,Hyderabad,0,1.0,9840.33,0.0,80.0,2026-06-12,0.0,Premium,98.0,3.0,1.0,32.733333,5.5,1.0,2025-04-21,515.0
2,CRM-00003,Meera Banerjee,mbanerjee62@yahoo.in,+91-97671-62008,"291, Indiranagar",Bengaluru,Ka,560086.0,2022-05-04,Mid-Market,Active,Bengaluru,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN


# 8. Churn Pattern Discovery

A quick look at what actually separates churners from retained customers, before we engineer
anything or train a model. This is descriptive only — **association, not causation**.

In [12]:
customer360["ticket_count"] = customer360["ticket_count"].fillna(0)
customer360["unresolved_ticket_count"] = customer360["unresolved_ticket_count"].fillna(0)
customer360["high_severity_count"] = customer360["high_severity_count"].fillna(0)
customer360["payment_failed_count"] = customer360["payment_failed_count"].fillna(0)
customer360["has_billing_record"] = customer360["billing_records"].notna().astype(int)
customer360["has_support_ticket"] = (customer360["ticket_count"] > 0).astype(int)

for cat_col in ["segment", "account_status", "plan_type", "region_office"]:
    if cat_col in customer360:
        rate = customer360.groupby(cat_col)["churned"].agg(["mean", "count"]).sort_values("mean", ascending=False)
        rate.columns = ["churn_rate", "n_customers"]
        print(f"\nChurn rate by {cat_col}:")
        display(rate)


Churn rate by segment:


,churn_rate,n_customers
segment,,
-,0.057143,35
Enterprise,0.044313,677
Mid-Market,0.040059,1348
Startup,0.039574,657
Smb,0.035334,1783



Churn rate by account_status:


,churn_rate,n_customers
account_status,,
Closed,1.00000,186
Active,0.00213,4225
Suspended,0.00000,339



Churn rate by plan_type:


,churn_rate,n_customers
plan_type,,
Basic,0.046709,942
Standard,0.042254,994
Premium,0.037092,674
Enterprise,0.029900,301
-,0.000000,13



Churn rate by region_office:


,churn_rate,n_customers
region_office,,
Gurugram,0.058486,872
Hyderabad,0.046595,558
Pune,0.044156,1155
Chennai,0.042802,771
Bengaluru,0.025431,1101
Kolkata,0.020478,293


In [13]:
customer360["tenure_days_raw"] = (SNAPSHOT_DATE - customer360["signup_date"]).dt.days

numeric_pattern_cols = [
    "tenure_days_raw", "monthly_amount_mean", "payment_failed_count",
    "ticket_count", "unresolved_ticket_count", "avg_resolution_hours",
    "avg_satisfaction", "high_severity_count", "days_since_last_payment", "days_since_last_ticket",
]
numeric_pattern_cols = [c for c in numeric_pattern_cols if c in customer360]

comparison = customer360.groupby("churned")[numeric_pattern_cols].median().T
comparison.columns = ["median_retained", "median_churned"]
print("Median value by churn status (a quick, robust effect-size check):")
display(comparison)

Median value by churn status (a quick, robust effect-size check):


,median_retained,median_churned
tenure_days_raw,1349.000,1163.000
monthly_amount_mean,3518.575,3373.535
payment_failed_count,0.000,0.000
ticket_count,0.000,1.000
unresolved_ticket_count,0.000,0.000
avg_resolution_hours,25.600,35.100
avg_satisfaction,4.000,2.000
high_severity_count,0.000,0.000
days_since_last_payment,93.000,296.000
days_since_last_ticket,422.000,296.000


**Reading the table above:** for each numeric signal, compare `median_churned` vs
`median_retained`. A large gap (e.g. churners having far fewer days of tenure, more unresolved
tickets, or more failed payments) is a candidate predictive signal — but it is only
*associated* with churn, not proven to *cause* it, and every one of these must still pass the
leakage audit below before being used as a model feature.

# 9. Leakage Audit

For every candidate feature we ask: *would this information actually have been available at
the moment we needed to make the prediction (i.e. before the customer churned)?*

`account_status` is checked programmatically below because CRM status fields commonly contain
values like `"Cancelled"`/`"Churned"` that would **directly encode the label** — that would not
be a predictive signal, it would be the answer sheet.

In [14]:
print("account_status categories found:")
print(customer360["account_status"].value_counts(dropna=False))

status_churn_rate = customer360.groupby("account_status")["churned"].mean()
print("\nChurn rate by account_status:")
print(status_churn_rate)

# Automatic, defensible rule: if any status category is almost perfectly aligned with the
# label (near 0% or near 100% churn) and covers a non-trivial group, it is very likely to be
# either leakage or a status set *by* the churn event itself -- exclude it from features.
suspicious_status = status_churn_rate[(status_churn_rate > 0.95) | (status_churn_rate < 0.02)]
suspicious_status = suspicious_status[customer360["account_status"].value_counts() >= 5]
ACCOUNT_STATUS_IS_LEAKY = len(suspicious_status) > 0
print("\naccount_status flagged as leakage risk:", ACCOUNT_STATUS_IS_LEAKY)

account_status categories found:
account_status
Active       4225
Suspended     339
Closed        186
Name: count, dtype: int64

Churn rate by account_status:
account_status
Active       0.00213
Closed       1.00000
Suspended    0.00000
Name: churned, dtype: float64

account_status flagged as leakage risk: True


### Leakage audit table

| Feature | Leakage risk | Reason | Keep / Drop |
|---|---|---|---|
| `account_status` | **Checked above, programmatically** | A CRM status field can be set to something like "Cancelled" *as a result of* churn, which would leak the label. Automatically excluded if the check above shows a near-perfect split. | Conditional (see code) |
| `signup_date` / `tenure_days` | Low | Set at signup, long before any churn decision. | Keep |
| `monthly_amount_mean`, `payment_failed_count`, `days_since_last_payment` | Low | Billing behaviour up to the snapshot date; no post-churn billing flag exists in the data. | Keep |
| `ticket_count`, `unresolved_ticket_count`, `avg_resolution_hours`, `avg_satisfaction`, `high_severity_count`, `days_since_last_ticket` | Low–Medium | Legitimate pre-churn signal (frustration/friction). **Limitation:** the data has no explicit churn *date*, so we cannot mathematically prove every ticket used here predates the churn decision for every customer — this is stated explicitly as a limitation rather than hidden. | Keep, with caveat |
| `cust_ref` / `reporter_email` / `email` / `full_name` / `billing_id` / `ticket_id` | High (identifiers) | Free identifiers/PII, not predictive signal, and a model could overfit to id-like strings. | Drop from model features |
| `last_payment_date`, `last_ticket_date`, `signup_date` (raw datetimes) | N/A | Only the *derived* day-counts are used as features; raw timestamps themselves are not fed to the model. | Drop raw dates, keep derived counts |

No column in this dataset directly states a cancellation date or post-churn retention action,
so beyond `account_status` there is no other obvious leakage source to exclude.

# 10. Feature Engineering


In [15]:
customer360["tenure_days"] = customer360["tenure_days_raw"]
customer360.loc[customer360["tenure_days"] < 0, "tenure_days"] = np.nan  # can't have signed up in the future

FEATURE_NUMERIC = [
    "tenure_days", "monthly_amount_mean", "payment_failed_count", "amount_negative_flag_count",
    "days_since_last_payment", "ticket_count", "unresolved_ticket_count",
    "avg_resolution_hours", "avg_satisfaction", "high_severity_count", "days_since_last_ticket",
    "has_billing_record", "has_support_ticket",
]
FEATURE_NUMERIC = [c for c in FEATURE_NUMERIC if c in customer360.columns]

FEATURE_CATEGORICAL = ["segment", "plan_type", "region_office"]
if not ACCOUNT_STATUS_IS_LEAKY:
    FEATURE_CATEGORICAL.append("account_status")
FEATURE_CATEGORICAL = [c for c in FEATURE_CATEGORICAL if c in customer360.columns]

print("Numeric features:", FEATURE_NUMERIC)
print("Categorical features:", FEATURE_CATEGORICAL)

Numeric features: ['tenure_days', 'monthly_amount_mean', 'payment_failed_count', 'amount_negative_flag_count', 'days_since_last_payment', 'ticket_count', 'unresolved_ticket_count', 'avg_resolution_hours', 'avg_satisfaction', 'high_severity_count', 'days_since_last_ticket', 'has_billing_record', 'has_support_ticket']
Categorical features: ['segment', 'plan_type', 'region_office']


# 11. Feature Quality Checks

In [16]:
quality = pd.DataFrame({
    "missing_%": (customer360[FEATURE_NUMERIC + FEATURE_CATEGORICAL].isna().mean() * 100).round(1),
    "n_unique": customer360[FEATURE_NUMERIC + FEATURE_CATEGORICAL].nunique(dropna=True),
})
display(quality)

# Drop any categorical feature with unmanageable cardinality (would blow up one-hot encoding
# for no real benefit in a 3-hour hackathon model).
HIGH_CARD_LIMIT = 30
high_card = [c for c in FEATURE_CATEGORICAL if customer360[c].nunique(dropna=True) > HIGH_CARD_LIMIT]
if high_card:
    print("Dropping high-cardinality categorical feature(s):", high_card)
    FEATURE_CATEGORICAL = [c for c in FEATURE_CATEGORICAL if c not in high_card]

# Drop constant / near-constant numeric features (zero signal, pure noise for the model).
near_constant = [c for c in FEATURE_NUMERIC if customer360[c].nunique(dropna=True) <= 1]
if near_constant:
    print("Dropping constant feature(s):", near_constant)
    FEATURE_NUMERIC = [c for c in FEATURE_NUMERIC if c not in near_constant]

FEATURE_COLS = FEATURE_NUMERIC + FEATURE_CATEGORICAL
print("\nFinal feature set:", FEATURE_COLS)

,missing_%,n_unique
tenure_days,0.8,2159
monthly_amount_mean,37.3,2912
payment_failed_count,0.0,3
amount_negative_flag_count,36.5,3
days_since_last_payment,36.5,134
ticket_count,0.0,11
unresolved_ticket_count,0.0,5
avg_resolution_hours,66.2,997
avg_satisfaction,67.0,88
high_severity_count,0.0,5



Final feature set: ['tenure_days', 'monthly_amount_mean', 'payment_failed_count', 'amount_negative_flag_count', 'days_since_last_payment', 'ticket_count', 'unresolved_ticket_count', 'avg_resolution_hours', 'avg_satisfaction', 'high_severity_count', 'days_since_last_ticket', 'has_billing_record', 'has_support_ticket', 'segment', 'plan_type', 'region_office']


# 12. Train / Test Split

**Decision: stratified split, not a temporal split.** A temporal split needs an event date to
split on; `churn_labels.csv` has no date column, and there is no reliable "prediction cutoff"
timestamp anywhere in the data. A temporal split here would be fabricated, not genuine. We
therefore use a stratified 80/20 split on the target so the ~4% churn rate is preserved in both
sets, which matters for imbalanced-data evaluation.

In [17]:
X = customer360[FEATURE_COLS]
y = customer360["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print(f"Train: n={len(X_train)}  churn_rate={y_train.mean():.2%}")
print(f"Test:  n={len(X_test)}  churn_rate={y_test.mean():.2%}")

Train: n=3800  churn_rate=4.11%
Test:  n=950  churn_rate=4.11%


# 13. Class Imbalance Strategy

Churn is ~4% of customers. **Decision: `class_weight="balanced"` in every model, not SMOTE.**

- SMOTE synthesizes fake minority-class rows; with a real class this small (~150 positives in
  the training set) synthetic interpolation risks manufacturing unrealistic customers and
  overstating performance — a common cause of misleadingly great-looking imbalanced models.
- `class_weight="balanced"` re-weights the existing real examples during training — no
  fabricated data, same effect of not letting the majority class dominate the loss.
- The evaluation itself does not rely on a single 0.5 threshold: PR-AUC and top-K ranking
  metrics (Section 16) are what actually matter for a rare-event targeting problem, and
  threshold choice is treated as a separate, explicit business decision (Section 18).

# 14. Preprocessing Pipeline & Baseline

All preprocessing (imputation, scaling, one-hot encoding) is fit **only on the training set**
inside an `sklearn` pipeline, so nothing about the test set leaks into preprocessing.

In [18]:
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), FEATURE_NUMERIC),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), FEATURE_CATEGORICAL),
])

# Trivial baseline: predict the training churn rate for everyone. This is the floor any real
# model must beat -- if it can't, the model adds no value.
baseline_proba_test = np.full(len(y_test), y_train.mean())

# 15. Model Training

Three models, as scoped for a 3-hour hackathon (no large hyperparameter search):

1. **Logistic Regression** — interpretable baseline model.
2. **Random Forest** — non-linear, handles interactions, still reasonably interpretable via
   importances.
3. **HistGradientBoostingClassifier** — usually the strongest of the three on tabular data.

In [19]:
logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
])
logreg.fit(X_train, y_train)

rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1,
    )),
])
rf.fit(X_train, y_train)

hgb = Pipeline([
    ("prep", preprocess),
    ("clf", HistGradientBoostingClassifier(
        max_iter=200, max_depth=4, learning_rate=0.05,
        class_weight="balanced", random_state=RANDOM_SEED,
    )),
])
hgb.fit(X_train, y_train)

print("Models trained: logreg, rf, hgb")

Models trained: logreg, rf, hgb


# 16. Evaluation

**Accuracy is not reported as a success metric.** With ~4% churn, a model predicting "no
churn" for everyone scores ~96% accuracy while catching zero churners — accuracy would reward
a useless model. Instead:

- **PR-AUC (Average Precision)** — the right ranking metric for rare positive classes; unlike
  ROC-AUC it is not inflated by the huge number of easy true negatives.
- **ROC-AUC** — reported alongside for completeness/comparability, not as the primary metric.
- **Precision / Recall / F1 @ 0.5** — reported for reference only; 0.5 is not assumed to be a
  good operating threshold (see Section 18).
- **Precision@K / Recall@K / Lift@K (K = 5%, 10%, 20%)** — the metrics that map directly onto
  "if the retention team can only contact the top K% highest-risk customers, how many actual
  churners do we catch, and how much better is that than contacting people at random?"

In [20]:
def topk_metrics(y_true, y_score, k_frac):
    y_true = np.asarray(y_true)
    n = len(y_true)
    k = max(1, int(np.ceil(n * k_frac)))
    top_idx = np.argsort(-y_score)[:k]
    tp = y_true[top_idx].sum()
    precision_k = tp / k
    base_rate = y_true.mean()
    recall_k = tp / y_true.sum() if y_true.sum() > 0 else np.nan
    lift_k = precision_k / base_rate if base_rate > 0 else np.nan
    return precision_k, recall_k, lift_k

def evaluate(name, proba, y_true, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    row = {
        "model": name,
        "PR_AUC": average_precision_score(y_true, proba),
        "ROC_AUC": roc_auc_score(y_true, proba),
        f"Precision@{threshold}": precision_score(y_true, pred, zero_division=0),
        f"Recall@{threshold}": recall_score(y_true, pred, zero_division=0),
        f"F1@{threshold}": f1_score(y_true, pred, zero_division=0),
    }
    for kf in [0.05, 0.10, 0.20]:
        p, r, l = topk_metrics(y_true, proba, kf)
        pct = int(kf * 100)
        row[f"Precision@{pct}%"] = p
        row[f"Recall@{pct}%"] = r
        row[f"Lift@{pct}%"] = l
    return row

candidates = {
    "baseline_majority_rate": baseline_proba_test,
    "logistic_regression": logreg.predict_proba(X_test)[:, 1],
    "random_forest": rf.predict_proba(X_test)[:, 1],
    "hist_gradient_boosting": hgb.predict_proba(X_test)[:, 1],
}

comparison_rows = [evaluate(name, proba, y_test) for name, proba in candidates.items()]
comparison_df = pd.DataFrame(comparison_rows).set_index("model").round(4)
display(comparison_df)

,PR_AUC,ROC_AUC,Precision@0.5,Recall@0.5,F1@0.5,Precision@5%,Recall@5%,Lift@5%,Precision@10%,Recall@10%,Lift@10%,Precision@20%,Recall@20%,Lift@20%
model,,,,,,,,,,,,,,
baseline_majority_rate,0.0411,0.5000,0.0000,0.0000,0.0000,0.0417,0.0513,1.0150,0.0316,0.0769,0.7692,0.0263,0.1282,0.6410
logistic_regression,0.6421,0.8669,0.2661,0.7436,0.3919,0.5417,0.6667,13.1944,0.3053,0.7436,7.4359,0.1632,0.7949,3.9744
random_forest,0.5569,0.9012,0.4576,0.6923,0.5510,0.5417,0.6667,13.1944,0.3158,0.7692,7.6923,0.1684,0.8205,4.1026
hist_gradient_boosting,0.6028,0.8928,0.3944,0.7179,0.5091,0.5000,0.6154,12.1795,0.3053,0.7436,7.4359,0.1684,0.8205,4.1026


In [21]:
cm_rows = []
for name, proba in candidates.items():
    pred = (proba >= 0.5).astype(int)
    cm = confusion_matrix(y_test, pred)
    cm_rows.append((name, cm))
    print(f"\n{name} confusion matrix @0.5  [[TN, FP], [FN, TP]]:")
    print(cm)


baseline_majority_rate confusion matrix @0.5  [[TN, FP], [FN, TP]]:
[[911   0]
 [ 39   0]]

logistic_regression confusion matrix @0.5  [[TN, FP], [FN, TP]]:
[[831  80]
 [ 10  29]]

random_forest confusion matrix @0.5  [[TN, FP], [FN, TP]]:
[[879  32]
 [ 12  27]]

hist_gradient_boosting confusion matrix @0.5  [[TN, FP], [FN, TP]]:
[[868  43]
 [ 11  28]]


In [22]:
# Model selection: PR-AUC is the primary ranking criterion for a rare-event problem, with
# Recall@10% as a tie-breaker (does the model actually surface churners a retention team could
# realistically act on within limited contact capacity).
model_objs = {"logistic_regression": logreg, "random_forest": rf, "hist_gradient_boosting": hgb}
ranking = comparison_df.drop(index="baseline_majority_rate").sort_values(
    ["PR_AUC", "Recall@10%"], ascending=False
)
display(ranking)

BEST_MODEL_NAME = ranking.index[0]
best_model = model_objs[BEST_MODEL_NAME]
print("Selected model:", BEST_MODEL_NAME)

,PR_AUC,ROC_AUC,Precision@0.5,Recall@0.5,F1@0.5,Precision@5%,Recall@5%,Lift@5%,Precision@10%,Recall@10%,Lift@10%,Precision@20%,Recall@20%,Lift@20%
model,,,,,,,,,,,,,,
logistic_regression,0.6421,0.8669,0.2661,0.7436,0.3919,0.5417,0.6667,13.1944,0.3053,0.7436,7.4359,0.1632,0.7949,3.9744
hist_gradient_boosting,0.6028,0.8928,0.3944,0.7179,0.5091,0.5000,0.6154,12.1795,0.3053,0.7436,7.4359,0.1684,0.8205,4.1026
random_forest,0.5569,0.9012,0.4576,0.6923,0.5510,0.5417,0.6667,13.1944,0.3158,0.7692,7.6923,0.1684,0.8205,4.1026


Selected model: logistic_regression


# 17. Calibration

Raw scores from tree ensembles in particular are often poorly calibrated (a predicted "0.8"
does not necessarily mean an 80% real-world chance of churn). We calibrate the selected model
with Platt scaling (`method="sigmoid"`), which is the safer choice given how few positive
examples (~150) are available — isotonic regression is more flexible but overfits easily on
this little data.

Calibrated probabilities matter here because a commercial team may use the score itself (not
just the rank) to prioritize contact effort or size an incentive.

In [23]:
calibrated_model = CalibratedClassifierCV(best_model, method="sigmoid", cv=5)
calibrated_model.fit(X_train, y_train)

proba_calibrated = calibrated_model.predict_proba(X_test)[:, 1]
proba_uncalibrated = candidates[BEST_MODEL_NAME]

brier_before = brier_score_loss(y_test, proba_uncalibrated)
brier_after = brier_score_loss(y_test, proba_calibrated)
print(f"Brier score before calibration: {brier_before:.4f}")
print(f"Brier score after calibration:  {brier_after:.4f}  (lower is better)")

frac_pos, mean_pred = calibration_curve(y_test, proba_calibrated, n_bins=10, strategy="quantile")
calibration_table = pd.DataFrame({"mean_predicted_prob": mean_pred, "observed_churn_rate": frac_pos})
display(calibration_table.round(3))

Brier score before calibration: 0.0867
Brier score after calibration:  0.0219  (lower is better)


,mean_predicted_prob,observed_churn_rate
0,0.001,0.032
1,0.003,0.000
2,0.004,0.000
3,0.006,0.000
4,0.008,0.011
5,0.013,0.011
6,0.020,0.011
7,0.028,0.011
8,0.041,0.032
9,0.310,0.305


**Reading the calibration table:** in a well-calibrated model, `mean_predicted_prob` and
`observed_churn_rate` should be close for every bin. Given the small number of positives in the
test set, expect some bin-to-bin noise — this is stated as a limitation rather than glossed
over. We do **not** claim perfect calibration.

# 18. Threshold Analysis

0.5 is not assumed to be the right operating point — it is one of many options evaluated below.
No stated business capacity/cost was given for this hackathon problem, so rather than
pretending one threshold is definitively "correct", we show the trade-off curve and default to
**ranked top-K targeting** (Section 16's Precision/Recall/Lift@K) as the primary way a
retention team should use this model, since it directly answers "who do we call first with the
contact budget we actually have" without requiring us to invent a threshold.

In [24]:
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
threshold_rows = []
for t in thresholds:
    pred = (proba_calibrated >= t).astype(int)
    threshold_rows.append({
        "threshold": t,
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "n_customers_flagged": int(pred.sum()),
        "pct_of_test_set": pred.mean(),
    })

threshold_df = pd.DataFrame(threshold_rows).round(3)
display(threshold_df)

,threshold,precision,recall,n_customers_flagged,pct_of_test_set
0,0.1,0.458,0.692,59,0.062
1,0.2,0.545,0.615,44,0.046
2,0.3,0.625,0.513,32,0.034
3,0.4,0.731,0.487,26,0.027
4,0.5,0.800,0.410,20,0.021
5,0.6,0.941,0.410,17,0.018
6,0.7,1.000,0.385,15,0.016


# 19. Feature Importance

Feature importance shows **predictive association, not causation.** We use the language
"important predictive signal" / "associated with higher predicted churn risk" — never "causes
churn".

In [25]:
feature_names = best_model.named_steps["prep"].get_feature_names_out()

logreg_coefs = pd.Series(
    logreg.named_steps["clf"].coef_[0],
    index=logreg.named_steps["prep"].get_feature_names_out(),
    name="logreg_coefficient",
).sort_values(key=np.abs, ascending=False)

print("Logistic regression coefficients (sign = direction, magnitude = strength):")
display(logreg_coefs.head(15).to_frame())

Logistic regression coefficients (sign = direction, magnitude = strength):


,logreg_coefficient
num__days_since_last_payment,1.733765
cat__segment_-,1.380199
cat__segment_Enterprise,-1.377544
cat__plan_type_Basic,-1.232561
num__avg_satisfaction,-1.114390
cat__plan_type_Enterprise,1.104092
num__has_billing_record,-0.973915
cat__plan_type_-,-0.676651
cat__region_office_Bengaluru,-0.661615
cat__region_office_Kolkata,-0.640102


In [26]:
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=15, random_state=RANDOM_SEED,
    scoring="average_precision", n_jobs=-1,
)
perm_importance = pd.Series(perm.importances_mean, index=FEATURE_COLS, name="permutation_importance_PRAUC")
perm_importance = perm_importance.sort_values(ascending=False)

print(f"Permutation importance of {BEST_MODEL_NAME} (drop in PR-AUC when a feature is shuffled):")
display(perm_importance.to_frame())

Permutation importance of logistic_regression (drop in PR-AUC when a feature is shuffled):


,permutation_importance_PRAUC
days_since_last_payment,0.565905
avg_satisfaction,0.086649
has_billing_record,0.026127
high_severity_count,0.005121
tenure_days,0.003537
monthly_amount_mean,0.001959
payment_failed_count,0.001224
amount_negative_flag_count,0.000648
has_support_ticket,-0.000153
region_office,-0.000534


**Sanity check.** For each top feature, ask: does this make business sense?

- Recency/behavioural signals (falling engagement, failed payments, unresolved tickets, low
  tenure) being near the top matches domain intuition about churn drivers.
- If a `region_office` or identifier-adjacent feature unexpectedly dominates, that is a signal
  to re-open the leakage audit rather than accept it at face value — geography rarely *drives*
  churn as strongly as behavioural signals do, and a surprising result is more likely a data
  artefact than a real effect.

# 20. Error Analysis — Where Does the Model Perform Worst?

In [27]:
error_df = X_test.copy()
error_df["y_true"] = y_test.values
error_df["y_proba"] = proba_calibrated
error_df["y_pred"] = (proba_calibrated >= 0.5).astype(int)
error_df["error_type"] = np.select(
    [
        (error_df["y_true"] == 1) & (error_df["y_pred"] == 0),
        (error_df["y_true"] == 0) & (error_df["y_pred"] == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)

print(error_df["error_type"].value_counts())

group_cols = [c for c in ["segment", "plan_type", "region_office"] if c in error_df.columns]
for g in group_cols:
    print(f"\nError-rate breakdown by {g}:")
    tab = (
        error_df.groupby(g)["error_type"]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .round(3)
    )
    display(tab)

if "tenure_days" in error_df.columns:
    error_df["tenure_band"] = pd.cut(
        error_df["tenure_days"], bins=[-1, 90, 365, 730, np.inf],
        labels=["0-3mo", "3-12mo", "1-2yr", "2yr+"],
    )
    print("\nError-rate breakdown by tenure band:")
    display(error_df.groupby("tenure_band")["error_type"].value_counts(normalize=True).unstack(fill_value=0).round(3))

error_type
correct           923
false_negative     23
false_positive      4
Name: count, dtype: int64

Error-rate breakdown by segment:


error_type,correct,false_negative,false_positive
segment,,,
-,1.000,0.000,0.000
Enterprise,0.937,0.056,0.008
Mid-Market,0.975,0.022,0.004
Smb,0.983,0.011,0.006
Startup,0.977,0.023,0.000



Error-rate breakdown by plan_type:


error_type,correct,false_negative,false_positive
plan_type,,,
-,1.000,0.000,0.000
Basic,0.983,0.017,0.000
Enterprise,1.000,0.000,0.000
Premium,0.978,0.007,0.015
Standard,0.971,0.019,0.010



Error-rate breakdown by region_office:


error_type,correct,false_negative,false_positive
region_office,,,
Bengaluru,0.972,0.028,0.000
Chennai,0.981,0.019,0.000
Gurugram,0.972,0.023,0.006
Hyderabad,0.972,0.018,0.009
Kolkata,0.964,0.036,0.000
Pune,0.967,0.025,0.008



Error-rate breakdown by tenure band:


error_type,correct,false_negative,false_positive
tenure_band,,,
0-3mo,1.000,0.000,0.000
3-12mo,0.949,0.051,0.000
1-2yr,0.994,0.006,0.000
2yr+,0.971,0.023,0.006


# 21. Business Cost of Errors

The dataset does not include a clean per-customer lifetime-value or contact-cost field, so
costs are described **qualitatively** rather than inventing monetary figures.

**False Positive** — model flags a customer as high-risk, but they were never going to churn.
Cost: wasted retention-team contact time, an unnecessary discount/incentive offered, and a
small risk of annoying a happy customer with an unsolicited "please don't leave" outreach.

**False Negative** — a customer churns and the model never flagged them.
Cost: a lost customer, lost recurring revenue (`monthly_amount_mean` gives an order-of-magnitude
sense of the per-customer amount at stake, even though we don't have full lifetime value), and
a missed chance to intervene.

**Why this shapes the modeling choices above:** in most subscription businesses a false
negative (a quiet churner nobody reached out to) is materially more expensive than a false
positive (one unnecessary phone call), which is exactly why we lead with **ranking/recall-
oriented metrics (PR-AUC, Recall@K)** rather than optimizing for precision or accuracy, and why
`class_weight="balanced"` was chosen over letting the model default to predicting "no churn"
for everyone.

# 22. Customer Risk Scoring — Final Output

Score every customer with a usable feature row, rank by predicted probability, and assign risk
bands from that ranking (top 10% = High, next 20% = Medium, rest = Low) — a ranking-based
banding is used rather than a single hard-coded probability cut, consistent with the threshold
discussion in Section 18.

In [28]:
X_all = customer360[FEATURE_COLS]
all_proba = calibrated_model.predict_proba(X_all)[:, 1]

output = pd.DataFrame({
    "cust_id": customer360["cust_id"],
    "churn_probability": all_proba,
})
output["rank"] = output["churn_probability"].rank(ascending=False, method="first").astype(int)
output = output.sort_values("rank").reset_index(drop=True)

n_customers = len(output)
top10_cut = int(np.ceil(n_customers * 0.10))
top30_cut = int(np.ceil(n_customers * 0.30))

def band(rank):
    if rank <= top10_cut:
        return "High"
    if rank <= top30_cut:
        return "Medium"
    return "Low"

output["risk_band"] = output["rank"].apply(band)

# Attach a few human-readable risk signals for the retention team.
signal_cols = [c for c in ["ticket_count", "unresolved_ticket_count", "payment_failed_count",
                            "days_since_last_payment", "tenure_days"] if c in customer360.columns]
output = output.merge(customer360[["cust_id"] + signal_cols], on="cust_id", how="left")

display(output.head(10))
print("\nRisk band sizes:")
print(output["risk_band"].value_counts())

OUTPUT_PATH = Path("customer_churn_predictions.csv")
output.to_csv(OUTPUT_PATH, index=False)
print("\nSaved:", OUTPUT_PATH.resolve())

,cust_id,churn_probability,rank,risk_band,ticket_count,unresolved_ticket_count,payment_failed_count,days_since_last_payment,tenure_days
0,CRM-04750,0.999971,1,High,2.0,0.0,0.0,620.0,645.0
1,CRM-00742,0.999900,2,High,8.0,1.0,0.0,466.0,2395.0
2,CRM-00314,0.999846,3,High,1.0,0.0,0.0,463.0,138.0
3,CRM-03787,0.999826,4,High,2.0,0.0,0.0,461.0,222.0
4,CRM-00592,0.999703,5,High,7.0,1.0,0.0,417.0,1980.0
5,CRM-04554,0.999682,6,High,6.0,1.0,0.0,453.0,1268.0
6,CRM-02884,0.999659,7,High,0.0,0.0,1.0,488.0,187.0
7,CRM-02244,0.999643,8,High,1.0,0.0,0.0,616.0,2457.0
8,CRM-03508,0.999590,9,High,1.0,0.0,0.0,479.0,1116.0
9,CRM-03301,0.999477,10,High,3.0,0.0,0.0,448.0,1719.0



Risk band sizes:
risk_band
Low       3325
Medium     950
High       475
Name: count, dtype: int64

Saved: C:\Users\LENOVO\Downloads\customer_churn_predictions.csv


# 23. Final Model Comparison

In [29]:
final_comparison = comparison_df[[
    "PR_AUC", "ROC_AUC", "Precision@0.5", "Recall@0.5", "F1@0.5",
    "Precision@10%", "Recall@10%", "Lift@10%",
]].round(4)
display(final_comparison)
print(f"\nSelected model: {BEST_MODEL_NAME} (highest PR-AUC among trained candidates, "
      f"Recall@10% used as tie-breaker). Calibration was then applied on top of this model.")

,PR_AUC,ROC_AUC,Precision@0.5,Recall@0.5,F1@0.5,Precision@10%,Recall@10%,Lift@10%
model,,,,,,,,
baseline_majority_rate,0.0411,0.5000,0.0000,0.0000,0.0000,0.0316,0.0769,0.7692
logistic_regression,0.6421,0.8669,0.2661,0.7436,0.3919,0.3053,0.7436,7.4359
random_forest,0.5569,0.9012,0.4576,0.6923,0.5510,0.3158,0.7692,7.6923
hist_gradient_boosting,0.6028,0.8928,0.3944,0.7179,0.5091,0.3053,0.7436,7.4359



Selected model: logistic_regression (highest PR-AUC among trained candidates, Recall@10% used as tie-breaker). Calibration was then applied on top of this model.


# 24. Final Executive Summary

In [30]:
best_row = comparison_df.loc[BEST_MODEL_NAME]
top_features_list = ", ".join(perm_importance.head(5).index.tolist())
n_high_risk = int((output["risk_band"] == "High").sum())

summary = f'''
CHURN MODEL -- EXECUTIVE SUMMARY
=================================
Customers analyzed (labelled, joined to CRM): {len(customer360)}
Overall churn rate: {churn_rate:.2%} ({churners} churners / {n} labelled customers)

Selected model: {BEST_MODEL_NAME} (calibrated with Platt scaling)
  PR-AUC (test):      {best_row['PR_AUC']:.3f}
  ROC-AUC (test):     {best_row['ROC_AUC']:.3f}
  Precision@10%:      {best_row['Precision@10%']:.3f}
  Recall@10%:         {best_row['Recall@10%']:.3f}
  Lift@10%:           {best_row['Lift@10%']:.2f}x over random targeting
  Brier score (calibrated): {brier_after:.4f}

Top predictive signals (permutation importance): {top_features_list}
  (association with predicted risk, not proven causation -- see Section 19)

Customers flagged High risk (top decile by rank): {n_high_risk}

Main limitations:
  - No explicit churn-event date exists in the data, so support/billing recency features
    could not be strictly proven to predate every customer's churn decision (see Section 9).
  - Billing links to CRM only via email; billing rows with a missing/unmatched email
    contribute no billing features for that customer.
  - Calibration uses Platt scaling on a modest number of positive examples in the training
    set; probability estimates are directionally useful but not claimed to be exact.
  - No explicit contact-capacity or cost figures were provided, so the threshold in Section 18
    is illustrative -- ranked top-K targeting (Section 16) is the recommended way to act on
    this model's output.
'''
print(summary)


CHURN MODEL -- EXECUTIVE SUMMARY
Customers analyzed (labelled, joined to CRM): 4750
Overall churn rate: 4.11% (195 churners / 4750 labelled customers)

Selected model: logistic_regression (calibrated with Platt scaling)
  PR-AUC (test):      0.642
  ROC-AUC (test):     0.867
  Precision@10%:      0.305
  Recall@10%:         0.744
  Lift@10%:           7.44x over random targeting
  Brier score (calibrated): 0.0219

Top predictive signals (permutation importance): days_since_last_payment, avg_satisfaction, has_billing_record, high_severity_count, tenure_days
  (association with predicted risk, not proven causation -- see Section 19)

Customers flagged High risk (top decile by rank): 475

Main limitations:
  - No explicit churn-event date exists in the data, so support/billing recency features
    could not be strictly proven to predate every customer's churn decision (see Section 9).
  - Billing links to CRM only via email; billing rows with a missing/unmatched email
    contribute no b